In [1]:
!pip install openai-whisper
!pip install librosa
!pip install scikit-learn
!pip install scipy
!pip install pydub
!pip install soundfile
!pip install silero-vad
!pip install resemblyzer
!pip install spectralcluster
!pip install demucs
!pip install nltk
!pip install nemo_toolkit[asr]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=0b459647f9f049ffdcb69d3627cf460d6615c52abb5227cfb7a3fd1227ef99cc
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 129.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 8.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 7.9 MB/s eta 0:00:00
  Created wheel for demucs: filename=demucs-4.0.1-py3-none-any.whl size=78388 sha256=a90e057f3c295177c880246b1398292e40806c8365150a225bbeb2997691eb70
  Stored in directory: /root/.cache/pip/wheels/1b/0c/20/a3b3daa1f9b65c8b0445729f94740ec335d0f86f1066c5c414
  Created wheel for julius: filename=julius-0.2.7-

In [3]:
import os
import json
import numpy as np
import torch
import whisper
import librosa
import soundfile as sf
from scipy.io import wavfile
from sklearn.cluster import AgglomerativeClustering
from spectralcluster import SpectralClusterer
from resemblyzer import VoiceEncoder, preprocess_wav
import torchaudio
from demucs.pretrained import get_model
from demucs.apply import apply_model
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from typing import Dict, List, Tuple, Optional
import warnings
from nemo.collections.asr.models import EncDecSpeakerLabelModel
warnings.filterwarnings('ignore')

class PipelineConfig:
    TARGET_SAMPLE_RATE = 16000
    AUDIO_FORMAT = "wav"
    FOREGROUND_BACKGROUND_SEPARATION = True
    DEMUCS_MODEL = "htdemucs"
    VAD_THRESHOLD = 0.5
    MIN_SPEECH_DURATION_MS = 250
    MIN_SILENCE_DURATION_MS = 100
    EMBEDDING_WINDOW_SEC = 1.5
    EMBEDDING_STEP_SEC = 0.75
    MIN_SPEAKERS = 1
    MAX_SPEAKERS = 10
    WHISPER_MODEL = "large-v3"
    WHISPER_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    APPLY_LEMMATIZATION = True
    APPLY_STEMMING = True
    OUTPUT_FORMAT = "json"
    INCLUDE_WORD_TIMESTAMPS = True

config = PipelineConfig()

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

[NeMo W 2025-12-06 06:39:32 nemo_logging:405] Megatron num_microbatches_calculator not found, using Apex version.


True

In [4]:
class PreprocessingExpert:
    def __init__(self, target_sr: int = 16000, use_separation: bool = True):
        self.target_sr = target_sr
        self.use_separation = use_separation
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        if self.use_separation:
            self.separator = get_model(name='htdemucs')
            self.separator.to(self.device)
            self.separator.eval()

    def process(self, audio_path: str) -> Tuple[np.ndarray, int]:
        if self.use_separation:
            waveform, sr = librosa.load(audio_path, sr=44100, mono=False)
            if waveform.ndim == 1:
                waveform = np.stack([waveform, waveform])
            waveform = self._separate_foreground(waveform, 44100)
            if sr != self.target_sr:
                waveform = librosa.resample(waveform, orig_sr=44100, target_sr=self.target_sr)
            sr = self.target_sr
        else:
            waveform, sr = librosa.load(audio_path, sr=self.target_sr, mono=True)

        if np.abs(waveform).max() > 0:
            waveform = waveform / np.abs(waveform).max()

        return waveform, sr

    def _separate_foreground(self, waveform: np.ndarray, sr: int) -> np.ndarray:
        audio_tensor = torch.tensor(waveform).unsqueeze(0).float().to(self.device)

        with torch.no_grad():
            sources = apply_model(self.separator, audio_tensor, device=self.device)

        vocals = sources[0, self.separator.sources.index('vocals')].mean(0).cpu().numpy()

        if np.abs(vocals).max() > 0:
            vocals = vocals / np.abs(vocals).max()

        return vocals

    def save_preprocessed(self, waveform: np.ndarray, sr: int, output_path: str):
        sf.write(output_path, waveform, sr)

preprocessing_expert = PreprocessingExpert(
    target_sr=config.TARGET_SAMPLE_RATE,
    use_separation=config.FOREGROUND_BACKGROUND_SEPARATION
)


In [5]:
class VADExpert:
    def __init__(self, threshold: float = 0.5,
                 min_speech_ms: int = 250,
                 min_silence_ms: int = 100):
        self.model, utils = torch.hub.load(
            repo_or_dir='snakers4/silero-vad',
            model='silero_vad',
            force_reload=False,
            onnx=False
        )
        self.get_speech_timestamps = utils[0]
        self.threshold = threshold
        self.min_speech_ms = min_speech_ms
        self.min_silence_ms = min_silence_ms

    def process(self, waveform: np.ndarray, sr: int) -> List[Dict]:
        audio_tensor = torch.FloatTensor(waveform)
        speech_timestamps = self.get_speech_timestamps(
            audio_tensor,
            self.model,
            threshold=self.threshold,
            sampling_rate=sr,
            min_speech_duration_ms=self.min_speech_ms,
            min_silence_duration_ms=self.min_silence_ms
        )

        segments = []
        for i, ts in enumerate(speech_timestamps):
            segment = {
                'segment_id': i,
                'start_sample': ts['start'],
                'end_sample': ts['end'],
                'start_time': ts['start'] / sr,
                'end_time': ts['end'] / sr,
                'duration': (ts['end'] - ts['start']) / sr,
                'waveform': waveform[ts['start']:ts['end']]
            }
            segments.append(segment)

        return segments

vad_expert = VADExpert(
    threshold=config.VAD_THRESHOLD,
    min_speech_ms=config.MIN_SPEECH_DURATION_MS,
    min_silence_ms=config.MIN_SILENCE_DURATION_MS
)


Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip


In [13]:
class DiarizationExpert:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        # Load NeMo Titanet
        self.speaker_model = EncDecSpeakerLabelModel.from_pretrained("titanet_large")
        self.speaker_model.eval().to(self.device)
        # Silero VAD (already loaded in VADExpert, but for completeness)
        self.model, self.utils = torch.hub.load('snakers4/silero-vad', 'silero_vad', force_reload=False)
        self.get_speech_timestamps = self.utils[0]

    def process(self, segments: List[Dict], sr: int, waveform: np.ndarray) -> List[Dict]:
        # Filter segments > 1.5s
        valid_segments = [seg for seg in segments if seg['duration'] > 1.5]

        if len(valid_segments) < 2:
            for seg in valid_segments:
                seg['speaker'] = "SPEAKER_00"
            return valid_segments

        import soundfile as sf
        import os
        temp_dir = "/tmp/nemo_segments"
        os.makedirs(temp_dir, exist_ok=True)

        # Extract NeMo embeddings
        embeddings = []
        for i, seg in enumerate(valid_segments):
            temp_file = f"{temp_dir}/seg_{i}.wav"
            start_sample = int(seg['start_time'] * sr)
            end_sample = int(seg['end_sample'])
            seg_audio = waveform[start_sample:end_sample]
            sf.write(temp_file, seg_audio, sr)

            emb = self.speaker_model.get_embedding(temp_file).cpu().numpy().squeeze()
            embeddings.append(emb)

        embeddings = np.array(embeddings)

        # Clustering with silhouette optimization
        from sklearn.cluster import AgglomerativeClustering
        from sklearn.metrics.pairwise import cosine_distances
        from sklearn.metrics import silhouette_score

        dist_matrix = cosine_distances(embeddings)

        best_n_clusters = 1
        best_score = -1
        best_labels = None

        for n in range(2, min(6, len(embeddings)//4 + 1)):
            clustering = AgglomerativeClustering(n_clusters=n, metric='precomputed', linkage='average')
            labels = clustering.fit_predict(dist_matrix)
            if len(np.unique(labels)) > 1:
                score = silhouette_score(dist_matrix, labels, metric='precomputed')
                print(f"Tried {n} clusters: silhouette={score:.3f}")
                if score > best_score:
                    best_score = score
                    best_n_clusters = n
                    best_labels = labels

        print(f"Identified {best_n_clusters} speakers with silhouette score: {best_score:.3f}")

        # Assign labels
        for seg, label in zip(valid_segments, best_labels):
            seg['speaker'] = f"SPEAKER_{int(label):02d}"

        # Merge consecutive
        return self._merge_consecutive_segments(valid_segments)

    def _merge_consecutive_segments(self, segments: List[Dict]) -> List[Dict]:
        if not segments:
            return []

        merged = []
        current = segments[0].copy()

        for seg in segments[1:]:
            if (seg['speaker'] == current['speaker'] and
                seg['start_time'] - current['end_time'] < 2.0):
                current['end_time'] = seg['end_time']
                current['end_sample'] = seg['end_sample']
                current['duration'] = current['end_time'] - current['start_time']
            else:
                merged.append(current)
                current = seg.copy()

        merged.append(current)
        for i, seg in enumerate(merged):
            seg['segment_id'] = i

        return merged
diarization_expert = DiarizationExpert()

[NeMo I 2025-12-06 06:42:54 nemo_logging:393] Downloading from: https://api.ngc.nvidia.com/v2/models/nvidia/nemo/titanet_large/versions/v1/files/titanet-l.nemo to /root/.cache/torch/NeMo/NeMo_2.6.0/titanet-l/11ba0924fdf87c049e339adbf6899d48/titanet-l.nemo
[NeMo I 2025-12-06 06:42:54 nemo_logging:393] Instantiating model from pre-trained checkpoint


[NeMo W 2025-12-06 06:42:54 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      speed:
        prob: 0.5
        sr: 16000
        resample_type: kaiser_fast
        min_speed_rate: 0.95
        max_speed_rate: 1.05
    num_workers: 15
    pin_memory: true
    
[NeMo W 2025-12-06 06:42:54 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data

[NeMo I 2025-12-06 06:42:55 nemo_logging:393] PADDING: 16
[NeMo I 2025-12-06 06:42:55 nemo_logging:393] Model EncDecSpeakerLabelModel was successfully restored from /root/.cache/torch/NeMo/NeMo_2.6.0/titanet-l/11ba0924fdf87c049e339adbf6899d48/titanet-l.nemo.


Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


In [7]:
class ASRExpert:
    def __init__(self, model_name: str = "large-v3", device: str = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = whisper.load_model(model_name, device=self.device)

    def process(self, segments: List[Dict], sr: int,
                full_waveform: np.ndarray) -> List[Dict]:
        for seg in segments:
            start_sample = int(seg['start_time'] * sr)
            end_sample = int(seg['end_time'] * sr)
            segment_audio = full_waveform[start_sample:end_sample]

            if len(segment_audio) < 400:
                segment_audio = np.pad(segment_audio, (0, 400 - len(segment_audio)))

            result = self.model.transcribe(
                segment_audio,
                language=None,
                word_timestamps=config.INCLUDE_WORD_TIMESTAMPS,
                verbose=False
            )

            seg['text'] = result['text'].strip()
            seg['language'] = result.get('language', 'unknown')

            if config.INCLUDE_WORD_TIMESTAMPS and 'segments' in result:
                words = []
                for segment in result['segments']:
                    if 'words' in segment:
                        for word in segment['words']:
                            words.append({
                                'word': word['word'].strip(),
                                'start': word['start'] + seg['start_time'],
                                'end': word['end'] + seg['start_time'],
                                'probability': word.get('probability', 1.0)
                            })
                seg['words'] = words

        return segments

asr_expert = ASRExpert(
    model_name=config.WHISPER_MODEL,
    device=config.WHISPER_DEVICE
)



100%|█████████████████████████████████████| 2.88G/2.88G [00:33<00:00, 92.2MiB/s]


In [8]:
class LinguisticExpert:
    def __init__(self, apply_lemmatization: bool = True, apply_stemming: bool = True):
        self.apply_lemmatization = apply_lemmatization
        self.apply_stemming = apply_stemming
        if self.apply_lemmatization:
            self.lemmatizer = WordNetLemmatizer()
        if self.apply_stemming:
            self.stemmer = PorterStemmer()

    def process(self, segments: List[Dict]) -> List[Dict]:
        for seg in segments:
            if 'text' not in seg or not seg['text']:
                continue

            tokens = word_tokenize(seg['text'].lower())

            if self.apply_lemmatization:
                lemmatized = [self.lemmatizer.lemmatize(token) for token in tokens]
                seg['lemmatized'] = ' '.join(lemmatized)

            if self.apply_stemming:
                stemmed = [self.stemmer.stem(token) for token in tokens]
                seg['stemmed'] = ' '.join(stemmed)

            if 'words' in seg:
                for word_info in seg['words']:
                    word_text = word_info['word'].lower().strip()
                    if self.apply_lemmatization:
                        word_info['lemmatized'] = self.lemmatizer.lemmatize(word_text)
                    if self.apply_stemming:
                        word_info['stemmed'] = self.stemmer.stem(word_text)

        return segments

linguistic_expert = LinguisticExpert(
    apply_lemmatization=config.APPLY_LEMMATIZATION,
    apply_stemming=config.APPLY_STEMMING
)


In [11]:
class UnificationExpert:
    def __init__(self):
        pass

    def process(self, segments: List[Dict], metadata: Dict) -> Dict:
        speakers = {}
        full_transcript = []

        for seg in segments:
            speaker = seg.get('speaker', 'UNKNOWN')
            text = seg.get('text', '')

            if speaker not in speakers:
                speakers[speaker] = {
                    'speaker_id': speaker,
                    'total_duration': 0.0,
                    'segments': []
                }

            speakers[speaker]['total_duration'] += seg['duration']
            speakers[speaker]['segments'].append({
                'segment_id': seg['segment_id'],
                'start_time': round(seg['start_time'], 3),
                'end_time': round(seg['end_time'], 3),
                'duration': round(seg['duration'], 3),
                'text': text,
                'lemmatized': seg.get('lemmatized', ''),
                'stemmed': seg.get('stemmed', '')
            })

            full_transcript.append({
                'segment_id': seg['segment_id'],
                'speaker': speaker,
                'start_time': round(seg['start_time'], 3),
                'end_time': round(seg['end_time'], 3),
                'text': text,
                'lemmatized': seg.get('lemmatized', ''),
                'stemmed': seg.get('stemmed', ''),
                'words': seg.get('words', [])
            })

        total_speech_duration = sum(seg['duration'] for seg in segments)

        output = {
            'metadata': {
                'audio_file': metadata.get('audio_file', 'unknown'),
                'sample_rate': metadata.get('sample_rate', 16000),
                'total_duration': round(metadata.get('total_duration', 0.0), 3),
                'total_speech_duration': round(total_speech_duration, 3),
                'num_segments': len(segments),
                'num_speakers': len(speakers),
                'language': metadata.get('language', 'auto-detected'),
                'model_versions': {
                    'vad': 'silero-vad',
                    'diarization': 'nemo_titanet_large',
                    'asr': config.WHISPER_MODEL
                }
            },
            'speakers': speakers,
            'transcript': full_transcript
        }

        return output

    def save_json(self, output: Dict, filepath: str):
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(output, f, indent=2, ensure_ascii=False)

    def save_text(self, output: Dict, filepath: str):
        with open(filepath, 'w', encoding='utf-8') as f:
            for item in output['transcript']:
                f.write(f"[{item['start_time']:.2f}s - {item['end_time']:.2f}s] ")
                f.write(f"{item['speaker']}: {item['text']}\n")

unification_expert = UnificationExpert()


In [14]:
class AudioAnalysisPipeline:
    def __init__(self):
        self.preprocessing = preprocessing_expert
        self.vad = vad_expert
        self.diarization = diarization_expert
        self.asr = asr_expert
        self.linguistic = linguistic_expert
        self.unification = unification_expert

    def process_audio(self, audio_path: str, output_dir: str = "./output") -> Dict:
        os.makedirs(output_dir, exist_ok=True)

        waveform, sr = self.preprocessing.process(audio_path)
        total_duration = len(waveform) / sr

        speech_segments = self.vad.process(waveform, sr)

        if len(speech_segments) == 0:
            return None

        diarized_segments = self.diarization.process(speech_segments, sr, waveform)
        transcribed_segments = self.asr.process(diarized_segments, sr, waveform)
        linguistic_segments = self.linguistic.process(transcribed_segments)

        metadata = {
            'audio_file': os.path.basename(audio_path),
            'sample_rate': sr,
            'total_duration': total_duration,
            'language': 'auto-detected'
        }

        output = self.unification.process(linguistic_segments, metadata)

        base_name = os.path.splitext(os.path.basename(audio_path))[0]
        json_path = os.path.join(output_dir, f"{base_name}_output.json")
        text_path = os.path.join(output_dir, f"{base_name}_transcript.txt")

        self.unification.save_json(output, json_path)
        self.unification.save_text(output, text_path)

        return output

pipeline = AudioAnalysisPipeline()


In [15]:
audio_file = "/content/ps6_01_214.wav"

if os.path.exists(audio_file):
    result = pipeline.process_audio(audio_file, output_dir="./output")

    if result:
        print(f"Total Duration: {result['metadata']['total_duration']:.2f}s")
        print(f"Speech Duration: {result['metadata']['total_speech_duration']:.2f}s")
        print(f"Number of Speakers: {result['metadata']['num_speakers']}")
        print(f"Number of Segments: {result['metadata']['num_segments']}")

        for item in result['transcript'][:5]:
            print(f"[{item['start_time']:.2f}s - {item['end_time']:.2f}s] {item['speaker']}: {item['text']}")

Tried 2 clusters: silhouette=0.524
Tried 3 clusters: silhouette=0.485
Tried 4 clusters: silhouette=0.510
Tried 5 clusters: silhouette=0.453
Identified 2 speakers with silhouette score: 0.524
Detected language: Punjabi


100%|██████████| 572/572 [00:11<00:00, 49.32frames/s]


Detected language: Punjabi


100%|██████████| 4674/4674 [01:11<00:00, 64.93frames/s]


Detected language: Punjabi


...

Detected language: Urdu


100%|██████████| 559/559 [00:03<00:00, 168.93frames/s]


Detected language: Punjabi


100%|██████████| 5956/5956 [01:08<00:00, 86.63frames/s]


Total Duration: 425.46s
Speech Duration: 416.77s
Number of Speakers: 2
Number of Segments: 12
[0.03s - 5.76s] SPEAKER_00: ਪੱਫੈਸ਼ਰ ਹਾਰਪਾਲ ਸਿਂਗ ਪਨੁ ਨੂ ਇਕ ਬੋਤ ਬਡਾ ਵੇਦਬਾਨ ਕੀਨੇ ਆਤਾਂ ਵੱ ਵੀ ਗਲਤ ਰਾਤ ਇਪਾਂ ਸਾਕਿਨ ।
[6.79s - 53.53s] SPEAKER_01: ਤੇ ਜਰਨੈਲ ਸਿਂਗ ਨੇ ਆਪਨਾ ਮੈਤਾ ਚੋਂਕ ਡੇਰਾ ਛਾਡ਼ਕੇ ਦਰਬਾਰ ਸਾਬ ਨੂ ਅਪਨਾ ਜਙਗ ਦਾ ਅਡਾ ਬਣਾਯਾ. ਇੰਨੇ ਸਾਰੀਆਂ ਗਲਾਂ ਦੇ ਬਰੁਂਦ ਆਜ ਤਾਕ ਹਰਪਾਲ ਸਿਂਗ ਨੇ ਬੋਲ ਸਕਿਆ. ਮਗਰੋ ਜੋ ਮਰਜੀ ਗਪਾ ਮਰੀਆਂ ਹੋਰਤੇ ਇਸਤੋ ਤਾਕ ਬੀਚ ਉਹਮਾਰਿਂ ਨੇ ਕੋਈ ਕਹਿਂਦ ਹੈ ਜਰਣੇਲ ਸਿਂਗ ਇਕੀ ਸਾਲੋ ਤੋ ਪਜਾਰੀਆ. ਤੇ ਫੇਰ ਦੂਜਿਏ ਲਵੇ ਖਾਡੇ ਹੁਂਦੇ ਨੇ ਉਦੇ ਜੀ ਉਤੇ ਸੋਆ ਗੋਡੀਆਂ ਲਗੀਆਂ. ਦੂਜਿਆ ਖਾਡੇ ਹੁਂਦੇ ਕੇਂਦਾ ਨੇ ਜੀ ਉਤੇ ਬਤੀ ਗੋਡੀਆਂ ਲਗੀਆਂ. ਦੂਜਿਆ ਖਾਡੇ ਲਗੀਆਂ ਲਗੀਆਂ ਲਗੀਆਂ.
[54.72s - 60.38s] SPEAKER_00: ਮੀਾ ਤੁਸੀ ਪਡੇ ਆਂਦੇ ਜੋ ਤੁਸੀ ਸੋਣੇ ਆਇਗਾ। ਉੱਦੇ ਅਕੋਡਿਂਗ ਤੁਹ ਮੀਨੂ ਲਗਦਾ ਤੁਸੀ ਸਾਰੀਏ ਕਲਾ ਛੂਟੀ ਕਰ ਰੇ ।
[61.51s - 66.65s] SPEAKER_01: ਮੈਂ ਪਾਈ ਜਰੇਲ ਸਿਂਗ ਦੇ ਅਮਰੇ ਸਾਮਰੇ ਹਡਾ ਹੋਕੇ ਸਵਾਲ ਜਵਾਬ ਕੀ ਤੇ ਰੇ ਪਰੇ ਆਕਾਠ ਦੇ ਵੀਕ ।
[68.64s - 91.13s] SPEAKER_00: ਪੱਫੈਸਰ ਹਰਪਾਲ ਸਿਂਗ ਪਨੁ ਨੂ ਏਕ ਬੋਤ ਬਡਾ ਵਿਦਵਾਣ ਗਿਣੇ ਆ ਜਾਂਦਾ. ਕੋਯੇ ਖੀਤੁ ਮੁਸਿ ਹੀਡੀ ਹੀਡੇ ਹੋਨੀ ਸਿਂਗ ਥੀਓ.


In [16]:
import gc

In [17]:
torch.cuda.empty_cache()
gc.collect()

40